# Stream Model To CPU

In this notebook we will demonstrate how to read tensors using the RunAI Model Streamer to the CPU memory and perform computation on their data.

The only requirement for running this notebook is a Linux machine.

## Preperation
We will start by downloading an example `.safetensors` file. Feel free to use your own.

In [1]:
import subprocess

url = 'https://huggingface.co/vidore/colpali/resolve/main/adapter_model.safetensors?download=true'
local_filename = 'model.safetensors'

wget_command = ['wget', '--content-disposition', url, '-O', local_filename]
subprocess.run(wget_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

## Streaming

To load the tensors from the file we need to create `SafetensorsStreamer` instance, perform the request, and start iterating over the tensors.

In [ ]:
from runai_model_streamer import SafetensorsStreamer

file_path = "model.safetensors"

def heavy_workload(tensor):
    # Perform heavy computation with the tensor
    # The computation is performed while the streamer 
    # continues reading the next tensors in the file
    return

with SafetensorsStreamer() as streamer:
    streamer.stream_file(file_path)
    for name, tensor in streamer.get_tensors():
        heavy_workload(tensor)

A heavy workload can be running on each tensor in the moment the tensor is yielded - in parallel to the CPP threads that continue reading from the storage. 

## Keeping tensors

Each tensor you are given belongs to you. It stays valid after the loop moves on and after the streamer closes, so you do not have to copy a tensor in order to keep it:

In [ ]:
tensors = {}

with SafetensorsStreamer() as streamer:
    streamer.stream_file(file_path)
    for name, tensor in streamer.get_tensors():
        tensors[name] = tensor

# The streamer has closed. The tensors still hold their own data.
first_name, first_tensor = next(iter(tensors.items()))
print(f"kept {len(tensors)} tensors; {first_name} has shape {tuple(first_tensor.shape)}")

Keeping every tensor, as above, holds the whole file in memory - which is what was asked for.

Keeping only a few is different. With a host destination, as here, the streamer hands over its read buffer whole, so one retained tensor keeps that entire buffer alive, not just its own bytes. If you know the subset you want in advance, ask for it with `tensor_names` and the rest is never read at all. If you only decide while iterating, copy the ones you want with `tensor.clone()` and let the rest go.

Pass `owned=False` to get the older behaviour, where a tensor is a view into a buffer the streamer reuses. Such a tensor must be used before the loop advances; keeping one past that point reads whatever was written next, with no error.